# 05 - Final Load Preparation

## RHoMIS Analytics — Section C, Group 7

This notebook converts the verified analysis from notebooks 03 and 04 into a final Tableau-ready dataset. The work here is intentionally practical:

- rebuild the approved derived analysis fields in one place
- lock the final KPI framework with clear formulas and valid populations
- keep only safe dashboard fields in the final export
- write the final Tableau-ready CSV

## 1. Setup

Load the cleaned ETL output and the household-level indicator file. The cleaned dataset stays the source of truth; this notebook creates an analysis-ready dashboard layer on top of it.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

### 1.1 Load the cleaned dataset and approved indicator file

This cell brings together the cleaned ETL output and the household-level indicator file so the final load notebook can rebuild the verified dashboard fields in one place.

In [2]:
data_candidates = [
    Path("../data/processed/rhomis_cleaned.csv.gz"),
    Path("data/processed/rhomis_cleaned.csv.gz"),
]
DATA_PATH = next((path for path in data_candidates if path.exists()), None)
assert DATA_PATH is not None, f"Missing cleaned dataset. Checked: {data_candidates}"

df = pd.read_csv(DATA_PATH, compression="gzip")
print(f"Loaded cleaned dataset: {df.shape[0]:,} rows x {df.shape[1]:,} columns")

indicator_candidates = [
    Path(".ai-workflow/research/indicator_data.tab"),
    Path("../.ai-workflow/research/indicator_data.tab"),
    Path("data/raw/indicator_data.tab"),
    Path("../data/raw/indicator_data.tab"),
]
INDICATOR_PATH = next((path for path in indicator_candidates if path.exists()), None)
assert INDICATOR_PATH is not None, (
    "Missing indicator_data.tab. Keep the household indicator file in .ai-workflow/research/ "
    "or data/raw/ before rerunning this notebook."
)

indicator_usecols = [
    "id_unique",
    "hh_size_mae",
    "livestock_tlu",
    "nr_months_food_shortage",
    "hfias_status",
    "fies_score",
    "hdds_good_season",
    "hdds_bad_season",
    "hdds_last_month",
    "total_income_lcu_per_year",
    "currency_conversion_lcu_to_ppp",
    "value_farm_products_consumed_lcu_per_hh_per_year",
    "farm_products_consumed_calories_kcal_per_hh_per_year",
    "proportion_of_value_controlled_female_adult",
    "proportion_of_value_controlled_male_adult",
]
indicators = pd.read_csv(INDICATOR_PATH, sep="\t", usecols=indicator_usecols)
indicators = indicators.rename(
    columns={
        "fies_score": "fies_score_indicator",
        "hfias_status": "hfias_status_indicator",
    }
)
df = df.merge(indicators, on="id_unique", how="left", validate="one_to_one")
print(f"Merged indicator data: {indicators.shape[1] - 1} fields")

Loaded cleaned dataset: 54,873 rows x 77 columns
Merged indicator data: 14 fields


## 2. Rebuild Final Analysis Fields

This section recreates the final fields needed for dashboarding so Tableau does not depend on notebook-specific intermediate state.

The design rule is simple:

- keep the cleaned data intact
- rebuild only the fields that were already justified in notebooks 03 and 04
- avoid raw local-currency metrics in the final dashboard export when a normalized alternative exists

In [3]:
fies_cols = [f"fies_{i}" for i in range(1, 9)]
hfias_cols = [f"hfias_{i}" for i in range(1, 10)]
crop_name_cols = [f"crop_name_{i}" for i in range(1, 6)]
harvest_cols = [f"crop_harvest_kg_per_year_{i}" for i in range(1, 4)]
crop_income_cols = [f"crop_income_per_year_{i}" for i in range(1, 4)]
livestock_income_cols = [f"livestock_sale_income_{i}" for i in range(1, 3)]
income_cols = crop_income_cols + livestock_income_cols

hfias_frequency_map = {
    "never": 0,
    "monthly": 1,
    "fewpermonth": 2,
    "weekly": 3,
    "fewperweek": 4,
    "daily": 5,
}
low_education_levels = {"no_school", "illiterate", "koranic"}

def count_distinct_non_null(values) -> int:
    cleaned = {
        str(value).strip().lower()
        for value in values
        if pd.notna(value) and str(value).strip() != ""
    }
    return len(cleaned)

def pct(valid_rows: int, total_rows: int) -> float:
    return round((valid_rows / total_rows) * 100, 1) if total_rows else np.nan

df["food_shortage_flag"] = np.where(
    df["foodshortagetime"].isin(["y", "n"]),
    (df["foodshortagetime"] == "y").astype(int),
    np.nan,
)
df["food_shortage_status"] = pd.Series(pd.NA, index=df.index, dtype="object")
df.loc[df["food_shortage_flag"] == 1, "food_shortage_status"] = "Food Shortage"
df.loc[df["food_shortage_flag"] == 0, "food_shortage_status"] = "No Food Shortage"

crop_name_diversity_count = df[crop_name_cols].apply(count_distinct_non_null, axis=1)
crop_count_numeric = pd.to_numeric(df["crop_count"], errors="coerce")
df["crop_diversity_count"] = crop_count_numeric.combine_first(crop_name_diversity_count).clip(lower=0)

harvest_numeric = df[harvest_cols].apply(pd.to_numeric, errors="coerce")
df["crop_harvest_total_kg_observed"] = harvest_numeric.where(harvest_numeric > 0).sum(axis=1, min_count=1)

df["crop_income_total_lcu_observed"] = df[crop_income_cols].apply(pd.to_numeric, errors="coerce").sum(axis=1, min_count=1)
df["livestock_income_total_lcu_observed"] = df[livestock_income_cols].apply(pd.to_numeric, errors="coerce").sum(axis=1, min_count=1)
df["farm_cash_income_lcu_observed"] = df[income_cols].apply(pd.to_numeric, errors="coerce").sum(axis=1, min_count=1)

df["total_income_ppp_per_year"] = np.where(
    (df["total_income_lcu_per_year"].notna()) & (df["currency_conversion_lcu_to_ppp"] > 0),
    df["total_income_lcu_per_year"] / df["currency_conversion_lcu_to_ppp"],
    np.nan,
)
df["income_ppp_per_mae"] = np.where(
    (df["total_income_ppp_per_year"].notna()) & (df["hh_size_mae"] > 0),
    df["total_income_ppp_per_year"] / df["hh_size_mae"],
    np.nan,
)
df["farm_value_consumed_ppp_per_year"] = np.where(
    (df["value_farm_products_consumed_lcu_per_hh_per_year"].notna())
    & (df["currency_conversion_lcu_to_ppp"] > 0),
    df["value_farm_products_consumed_lcu_per_hh_per_year"] / df["currency_conversion_lcu_to_ppp"],
    np.nan,
)

df["country_income_rank_pct"] = df.groupby("country")["total_income_ppp_per_year"].rank(pct=True)
df["country_income_quintile"] = pd.cut(
    df["country_income_rank_pct"],
    bins=[0.0, 0.2, 0.4, 0.6, 0.8, 1.0],
    labels=["Lowest", "Low", "Middle", "High", "Highest"],
    include_lowest=True,
)

df["land_productivity_kg_per_ha"] = np.where(
    (df["crop_harvest_total_kg_observed"] > 0) & (df["land_cultivated_ha"] > 0),
    df["crop_harvest_total_kg_observed"] / df["land_cultivated_ha"],
    np.nan,
)

productivity_non_null = df["land_productivity_kg_per_ha"].dropna()
df["productivity_quartile"] = pd.Series(pd.NA, index=df.index, dtype="object")
if not productivity_non_null.empty:
    df.loc[productivity_non_null.index, "productivity_quartile"] = pd.qcut(
        productivity_non_null,
        q=4,
        labels=["Q1 lowest", "Q2", "Q3", "Q4 highest"],
        duplicates="drop",
    ).astype(str)

fies_valid_response = df[fies_cols].isin(["y", "n"]).all(axis=1)
fies_count_from_module = np.where(fies_valid_response, df[fies_cols].eq("y").sum(axis=1), np.nan)
df["fies_yes_count"] = df["fies_score_indicator"].combine_first(pd.Series(fies_count_from_module, index=df.index))

hfias_valid_response = df[hfias_cols].isin(hfias_frequency_map.keys()).all(axis=1)
df["hfias_frequency_score"] = np.where(
    hfias_valid_response,
    df[hfias_cols].apply(
        lambda row: sum(hfias_frequency_map.get(value, np.nan) for value in row),
        axis=1,
    ),
    np.nan,
)

df["irrigated_flag"] = np.where(
    df["land_irrigated"].isin(["y", "n"]),
    (df["land_irrigated"] == "y").astype(int),
    np.nan,
)
df["offfarm_flag"] = np.where(
    df["offfarm_incomes_any"].isin(["y", "n"]),
    (df["offfarm_incomes_any"] == "y").astype(int),
    np.nan,
)
df["female_respondent_flag"] = np.where(
    df["respondentsex"].isin(["f", "m"]),
    (df["respondentsex"] == "f").astype(int),
    np.nan,
)
df["low_education_flag"] = np.where(
    df["education_level"].notna(),
    df["education_level"].isin(low_education_levels).astype(int),
    np.nan,
)

### 2.1 Recreate the final vulnerability segmentation

The dashboard needs an interpretable household profile field, so this block rebuilds the clustering-based segmentation and maps the clusters to clear profile names.

In [4]:
seg_df = df[
    [
        "food_shortage_flag",
        "household_size_derived",
        "crop_diversity_count",
        "land_cultivated_ha",
        "land_productivity_kg_per_ha",
        "country_income_rank_pct",
        "irrigated_flag",
    ]
].dropna().copy()

seg_df = seg_df[
    (seg_df["land_cultivated_ha"] > 0) & (seg_df["land_productivity_kg_per_ha"] > 0)
].copy()

seg_df["log_land_cultivated"] = np.log1p(seg_df["land_cultivated_ha"])
seg_df["log_land_productivity"] = np.log1p(seg_df["land_productivity_kg_per_ha"])

cluster_features = [
    "household_size_derived",
    "crop_diversity_count",
    "log_land_cultivated",
    "log_land_productivity",
    "country_income_rank_pct",
    "irrigated_flag",
]
scaler = StandardScaler()
X = scaler.fit_transform(seg_df[cluster_features])

silhouette_rows = []
labels_by_k = {}
sample_idx = seg_df.sample(n=min(8000, len(seg_df)), random_state=42).index
sample_positions = seg_df.index.get_indexer(sample_idx)
X_sample = X[sample_positions]

for k in range(3, 7):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = kmeans.fit_predict(X)
    labels_by_k[k] = labels
    silhouette_rows.append(
        {
            "k": k,
            "silhouette": round(
                float(silhouette_score(X_sample, labels[sample_positions])),
                4,
            ),
        }
    )

silhouette_df = pd.DataFrame(silhouette_rows)
chosen_k = 4
seg_df["cluster"] = labels_by_k[chosen_k]

cluster_summary = seg_df.groupby("cluster").agg(
    households=("cluster", "size"),
    food_shortage_rate=("food_shortage_flag", "mean"),
    household_size=("household_size_derived", "median"),
    crop_diversity=("crop_diversity_count", "median"),
    land_ha=("land_cultivated_ha", "median"),
    prod_kg_ha=("land_productivity_kg_per_ha", "median"),
    income_rank=("country_income_rank_pct", "median"),
    irrigated_share=("irrigated_flag", "mean"),
).round(3)
cluster_summary["food_shortage_rate"] = (cluster_summary["food_shortage_rate"] * 100).round(1)
cluster_summary["irrigated_share"] = (cluster_summary["irrigated_share"] * 100).round(1)

def assign_profile_name(row) -> str:
    if row["irrigated_share"] >= 80:
        return "Irrigated higher-productivity"
    if row["food_shortage_rate"] >= 75 and row["income_rank"] <= 0.35 and row["prod_kg_ha"] < 500:
        return "Low-income low-productivity rainfed"
    if row["household_size"] >= 10 and row["land_ha"] >= 5:
        return "Large-household extensive low-productivity"
    return "Diversified rainfed better-off"

cluster_summary["profile_name"] = cluster_summary.apply(assign_profile_name, axis=1)
profile_map = cluster_summary["profile_name"].to_dict()
seg_df["vulnerability_profile_k4"] = seg_df["cluster"].map(profile_map)

df["vulnerability_profile_k4"] = pd.Series(pd.NA, index=df.index, dtype="object")
df.loc[seg_df.index, "vulnerability_profile_k4"] = seg_df["vulnerability_profile_k4"]

display(silhouette_df)
display(cluster_summary.reset_index().sort_values("food_shortage_rate", ascending=False))

,k,silhouette
0,3,0.236
1,4,0.221
2,5,0.222
3,6,0.216


,cluster,households,food_shortage_rate,household_size,crop_diversity,land_ha,prod_kg_ha,income_rank,irrigated_share,profile_name
2,2,14330,78.700,6.000,2.000,1.500,316.667,0.242,0.300,Low-income low-productivity rainfed
1,1,6860,72.300,13.000,2.000,8.000,230.316,0.683,4.400,Large-household extensive low-productivity
0,0,12855,62.700,5.000,3.000,1.295,"1,300.000",0.696,0.000,Diversified rainfed better-off
3,3,7753,55.800,6.000,3.000,1.214,"1,375.000",0.601,100.000,Irrigated higher-productivity


## 3. Coverage And Validation

Before exporting anything, confirm that the final dashboard fields are populated on the right populations. This is the guardrail that keeps the Tableau layer honest.

In [5]:
total_rows = len(df)

coverage_fields = [
    "food_shortage_flag",
    "crop_diversity_count",
    "land_productivity_kg_per_ha",
    "total_income_ppp_per_year",
    "income_ppp_per_mae",
    "country_income_rank_pct",
    "fies_yes_count",
    "hfias_status_indicator",
    "hfias_frequency_score",
    "hdds_bad_season",
    "vulnerability_profile_k4",
]

coverage_df = pd.DataFrame(
    [
        {
            "field": field,
            "valid_rows": int(df[field].notna().sum()),
            "pct_of_dataset": pct(int(df[field].notna().sum()), total_rows),
        }
        for field in coverage_fields
    ]
)
display(coverage_df)

validation_preview = df[
    [
        "food_shortage_flag",
        "food_shortage_status",
        "crop_diversity_count",
        "land_productivity_kg_per_ha",
        "income_ppp_per_mae",
        "country_income_rank_pct",
        "fies_yes_count",
        "hfias_frequency_score",
        "vulnerability_profile_k4",
    ]
].head(10)
display(validation_preview)

,field,valid_rows,pct_of_dataset
0,food_shortage_flag,47399,86.400
1,crop_diversity_count,54873,100.000
2,land_productivity_kg_per_ha,47803,87.100
3,total_income_ppp_per_year,50964,92.900
4,income_ppp_per_mae,48787,88.900
5,country_income_rank_pct,50964,92.900
6,fies_yes_count,25072,45.700
7,hfias_status_indicator,54873,100.000
8,hfias_frequency_score,6847,12.500
9,hdds_bad_season,34005,62.000


,food_shortage_flag,food_shortage_status,crop_diversity_count,land_productivity_kg_per_ha,income_ppp_per_mae,country_income_rank_pct,fies_yes_count,hfias_frequency_score,vulnerability_profile_k4
0,1.000,Food Shortage,3.000,197.684,46.990,0.508,6.000,NaN,Diversified rainfed better-off
1,1.000,Food Shortage,2.000,"2,141.578",2.424,0.316,8.000,NaN,Low-income low-productivity rainfed
2,1.000,Food Shortage,2.000,247.105,0.000,0.152,7.000,NaN,Low-income low-productivity rainfed
3,1.000,Food Shortage,3.000,617.763,113.345,0.603,3.000,NaN,Diversified rainfed better-off
4,1.000,Food Shortage,3.000,420.079,5.368,0.326,7.000,NaN,Low-income low-productivity rainfed
5,1.000,Food Shortage,3.000,"2,322.789",6.119,0.347,6.000,NaN,Diversified rainfed better-off
6,1.000,Food Shortage,3.000,494.210,3.831,0.325,6.000,NaN,Low-income low-productivity rainfed
7,1.000,Food Shortage,2.000,123.553,6.001,0.321,5.000,NaN,Low-income low-productivity rainfed
8,1.000,Food Shortage,2.000,543.631,0.000,0.152,5.000,NaN,Low-income low-productivity rainfed
9,1.000,Food Shortage,3.000,642.473,12.528,0.380,5.000,NaN,Low-income low-productivity rainfed


## 4. Final KPI Framework

These are the final dashboard KPIs that survived the analysis process. They are deliberately simple to explain, but they are not generic:

- the headline KPI is the verified food-shortage outcome
- land productivity and irrigation stay because they were strong structural drivers
- income is represented with PPP-normalized and within-country-safe measures
- FIES stays as a supporting severity layer on its valid subset

In [6]:
def fmt_value(name: str, value: float) -> str:
    if pd.isna(value):
        return "NA"
    if "Rate" in name or "Share" in name:
        return f"{value * 100:.1f}%"
    if "Count" in name:
        return f"{int(round(value)):,}"
    if "Income" in name or "Productivity" in name:
        return f"{value:,.2f}"
    return f"{value:,.2f}"

kpi_framework = pd.DataFrame(
    [
        {
            "kpi_name": "Food Shortage Rate",
            "python_formula": "df['food_shortage_flag'].mean()",
            "tableau_formula": "AVG([food_shortage_flag])",
            "valid_population": "Rows where foodshortagetime is y or n",
            "dashboard_role": "Primary headline KPI",
            "caution": "Always display the valid household count beside the rate",
        },
        {
            "kpi_name": "Food-Shortage Household Count",
            "python_formula": "(df['food_shortage_flag'] == 1).sum()",
            "tableau_formula": "COUNTD(IF [food_shortage_flag] = 1 THEN [id_unique] END)",
            "valid_population": "Rows where foodshortagetime is y or n",
            "dashboard_role": "Scale-of-need KPI",
            "caution": "Interpret with the rate, not on its own",
        },
        {
            "kpi_name": "Median Months of Food Shortage",
            "python_formula": "df['nr_months_food_shortage'].median()",
            "tableau_formula": "MEDIAN([nr_months_food_shortage])",
            "valid_population": "Rows with indicator-derived months of food shortage",
            "dashboard_role": "Headline severity KPI",
            "caution": "Use as a severity complement to the binary food-shortage rate",
        },
        {
            "kpi_name": "Irrigation Access Rate",
            "python_formula": "df['irrigated_flag'].mean()",
            "tableau_formula": "AVG([irrigated_flag])",
            "valid_population": "Rows where land_irrigated is y or n",
            "dashboard_role": "Intervention-readiness KPI",
            "caution": "Association does not imply causation",
        },
        {
            "kpi_name": "Median Land Productivity",
            "python_formula": "df['land_productivity_kg_per_ha'].median()",
            "tableau_formula": "MEDIAN([land_productivity_kg_per_ha])",
            "valid_population": "Rows with positive harvest and cultivated land",
            "dashboard_role": "Primary structural driver KPI",
            "caution": "Use medians or log-scaled charts because the distribution is very skewed",
        },
        {
            "kpi_name": "Median Income per MAE (PPP)",
            "python_formula": "df['income_ppp_per_mae'].median()",
            "tableau_formula": "MEDIAN([income_ppp_per_mae])",
            "valid_population": "Rows with valid income, PPP conversion, and MAE",
            "dashboard_role": "Normalized welfare KPI",
            "caution": "Do not replace this with raw local-currency averages",
        },
        {
            "kpi_name": "Lowest-Income Household Share",
            "python_formula": "(df['country_income_rank_pct'] <= 0.2).mean()",
            "tableau_formula": "AVG(IF [country_income_rank_pct] <= 0.20 THEN 1 ELSE 0 END)",
            "valid_population": "Rows with valid within-country income rank",
            "dashboard_role": "Targeting KPI",
            "caution": "This is a within-country welfare measure, not an absolute poverty rate",
        },
        {
            "kpi_name": "Mean FIES Score",
            "python_formula": "df['fies_yes_count'].mean()",
            "tableau_formula": "AVG([fies_yes_count])",
            "valid_population": "Rows with valid FIES score",
            "dashboard_role": "Supporting food-security severity KPI",
            "caution": "Subset metric only; never present as full-dataset coverage",
        },
    ]
)

kpi_values = {
    "Food Shortage Rate": df["food_shortage_flag"].mean(),
    "Food-Shortage Household Count": (df["food_shortage_flag"] == 1).sum(),
    "Median Months of Food Shortage": df["nr_months_food_shortage"].median(),
    "Irrigation Access Rate": df["irrigated_flag"].mean(),
    "Median Land Productivity": df["land_productivity_kg_per_ha"].median(),
    "Median Income per MAE (PPP)": df["income_ppp_per_mae"].median(),
    "Lowest-Income Household Share": (df["country_income_rank_pct"] <= 0.2).mean(),
    "Mean FIES Score": df["fies_yes_count"].mean(),
}

kpi_valid_rows = {
    "Food Shortage Rate": int(df["food_shortage_flag"].notna().sum()),
    "Food-Shortage Household Count": int(df["food_shortage_flag"].notna().sum()),
    "Median Months of Food Shortage": int(df["nr_months_food_shortage"].notna().sum()),
    "Irrigation Access Rate": int(df["irrigated_flag"].notna().sum()),
    "Median Land Productivity": int(df["land_productivity_kg_per_ha"].notna().sum()),
    "Median Income per MAE (PPP)": int(df["income_ppp_per_mae"].notna().sum()),
    "Lowest-Income Household Share": int(df["country_income_rank_pct"].notna().sum()),
    "Mean FIES Score": int(df["fies_yes_count"].notna().sum()),
}

kpi_results = pd.DataFrame(
    [
        {
            "kpi_name": name,
            "current_value": fmt_value(name, value),
            "valid_rows": kpi_valid_rows[name],
            "pct_of_dataset": pct(kpi_valid_rows[name], total_rows),
        }
        for name, value in kpi_values.items()
    ]
)

display(kpi_framework)
display(kpi_results)

,kpi_name,python_formula,tableau_formula,valid_population,dashboard_role,caution
0,Food Shortage Rate,df['food_shortage_flag'].mean(),AVG([food_shortage_flag]),Rows where foodshortagetime is y or n,Primary headline KPI,Always display the valid household count beside the rate
1,Food-Shortage Household Count,(df['food_shortage_flag'] == 1).sum(),COUNTD(IF [food_shortage_flag] = 1 THEN [id_unique] END),Rows where foodshortagetime is y or n,Scale-of-need KPI,"Interpret with the rate, not on its own"
2,Median Months of Food Shortage,df['nr_months_food_shortage'].median(),MEDIAN([nr_months_food_shortage]),Rows with indicator-derived months of food shortage,Headline severity KPI,Use as a severity complement to the binary food-shortage rate
3,Irrigation Access Rate,df['irrigated_flag'].mean(),AVG([irrigated_flag]),Rows where land_irrigated is y or n,Intervention-readiness KPI,Association does not imply causation
4,Median Land Productivity,df['land_productivity_kg_per_ha'].median(),MEDIAN([land_productivity_kg_per_ha]),Rows with positive harvest and cultivated land,Primary structural driver KPI,Use medians or log-scaled charts because the distribution is very skewed
5,Median Income per MAE (PPP),df['income_ppp_per_mae'].median(),MEDIAN([income_ppp_per_mae]),"Rows with valid income, PPP conversion, and MAE",Normalized welfare KPI,Do not replace this with raw local-currency averages
6,Lowest-Income Household Share,(df['country_income_rank_pct'] <= 0.2).mean(),AVG(IF [country_income_rank_pct] <= 0.20 THEN 1 ELSE 0 END),Rows with valid within-country income rank,Targeting KPI,"This is a within-country welfare measure, not an absolute poverty rate"
7,Mean FIES Score,df['fies_yes_count'].mean(),AVG([fies_yes_count]),Rows with valid FIES score,Supporting food-security severity KPI,Subset metric only; never present as full-dataset coverage


,kpi_name,current_value,valid_rows,pct_of_dataset
0,Food Shortage Rate,67.7%,47399,86.400
1,Food-Shortage Household Count,"32,097",47399,86.400
2,Median Months of Food Shortage,3.00,28537,52.000
3,Irrigation Access Rate,20.2%,50149,91.400
4,Median Land Productivity,600.00,47803,87.100
5,Median Income per MAE (PPP),123.30,48787,88.900
6,Lowest-Income Household Share,19.1%,50964,92.900
7,Mean FIES Score,3.69,25072,45.700


## 5. Final Tableau Field Set

The export keeps fields that are useful, explainable, and safe for dashboarding. Raw local-currency income fields are intentionally excluded so the Tableau layer does not encourage invalid cross-country comparisons.

In [7]:
export_columns = [
    "id_unique",
    "country",
    "iso_country_code",
    "year",
    "region",
    "gps_lat_rounded",
    "gps_lon_rounded",
    "foodshortagetime",
    "food_shortage_flag",
    "food_shortage_status",
    "nr_months_food_shortage",
    "household_size_derived",
    "hh_size_mae",
    "respondentsex",
    "female_respondent_flag",
    "education_level",
    "low_education_flag",
    "respondent_is_head",
    "land_cultivated_ha",
    "land_tenure",
    "land_ownership",
    "land_irrigated",
    "irrigated_flag",
    "farm_labour",
    "crop_diversity_count",
    "crop_harvest_total_kg_observed",
    "land_productivity_kg_per_ha",
    "productivity_quartile",
    "offfarm_incomes_any",
    "offfarm_flag",
    "total_income_ppp_per_year",
    "income_ppp_per_mae",
    "country_income_rank_pct",
    "country_income_quintile",
    "fies_yes_count",
    "hfias_status_indicator",
    "hfias_frequency_score",
    "hdds_good_season",
    "hdds_bad_season",
    "hdds_last_month",
    "livestock_tlu",
    "farm_value_consumed_ppp_per_year",
    "farm_products_consumed_calories_kcal_per_hh_per_year",
    "proportion_of_value_controlled_female_adult",
    "proportion_of_value_controlled_male_adult",
    "vulnerability_profile_k4",
]

tableau_df = df[export_columns].copy()
print(f"Final Tableau dataset preview: {tableau_df.shape[0]:,} rows x {tableau_df.shape[1]:,} columns")
display(tableau_df.head())

tableau_coverage = pd.DataFrame(
    [
        {
            "field": column,
            "valid_rows": int(tableau_df[column].notna().sum()),
            "pct_of_dataset": pct(int(tableau_df[column].notna().sum()), total_rows),
        }
        for column in [
            "food_shortage_flag",
            "nr_months_food_shortage",
            "irrigated_flag",
            "land_productivity_kg_per_ha",
            "income_ppp_per_mae",
            "country_income_rank_pct",
            "fies_yes_count",
            "hfias_frequency_score",
            "vulnerability_profile_k4",
        ]
    ]
)
display(tableau_coverage)

Final Tableau dataset preview: 54,873 rows x 46 columns


,id_unique,country,iso_country_code,year,region,gps_lat_rounded,gps_lon_rounded,foodshortagetime,food_shortage_flag,food_shortage_status,nr_months_food_shortage,household_size_derived,hh_size_mae,respondentsex,female_respondent_flag,education_level,low_education_flag,respondent_is_head,land_cultivated_ha,land_tenure,land_ownership,land_irrigated,irrigated_flag,farm_labour,crop_diversity_count,crop_harvest_total_kg_observed,land_productivity_kg_per_ha,productivity_quartile,offfarm_incomes_any,offfarm_flag,total_income_ppp_per_year,income_ppp_per_mae,country_income_rank_pct,country_income_quintile,fies_yes_count,hfias_status_indicator,hfias_frequency_score,hdds_good_season,hdds_bad_season,hdds_last_month,livestock_tlu,farm_value_consumed_ppp_per_year,farm_products_consumed_calories_kcal_per_hh_per_year,proportion_of_value_controlled_female_adult,proportion_of_value_controlled_male_adult,vulnerability_profile_k4
0,bf_adn_2019_1_1,burkina_faso,bf,2019,NaN,11.200,-1.000,y,1.000,Food Shortage,4.000,7,5.560,m,0.000,no_school,1.000,y,2.023,own_land rent_in_land,male_adult,n,0.000,household reciprocal,3.000,400.000,197.684,Q1 lowest,y,1.000,261.267,46.990,0.508,Middle,6.000,food_secure,NaN,5.000,2.000,NaN,1.020,82.220,"1,530,900.000",0.321,0.679,Diversified rainfed better-off
1,bf_adn_2019_2_1,burkina_faso,bf,2019,NaN,11.200,-1.000,y,1.000,Food Shortage,3.000,10,8.020,f,1.000,no_school,1.000,n,1.214,rent_in_land,NaN,n,0.000,household reciprocal,2.000,"2,600.000","2,141.578",Q4 highest,n,0.000,19.443,2.424,0.316,Low,8.000,food_secure,NaN,4.000,2.000,NaN,2.300,297.041,"6,743,100.000",0.469,0.531,Low-income low-productivity rainfed
2,bf_adn_2019_3_1,burkina_faso,bf,2019,NaN,11.200,-1.000,y,1.000,Food Shortage,4.000,6,4.690,f,1.000,no_school,1.000,n,0.809,own_land rent_in_land,male_adult,n,0.000,household reciprocal,2.000,200.000,247.105,Q2,n,0.000,0.000,0.000,0.152,Lowest,7.000,food_secure,NaN,4.000,2.000,NaN,0.300,24.709,"652,400.000",0.328,0.672,Low-income low-productivity rainfed
3,bf_adn_2019_4_1,burkina_faso,bf,2019,NaN,11.200,-1.000,y,1.000,Food Shortage,4.000,6,4.460,f,1.000,no_school,1.000,n,0.809,communal_land,NaN,n,0.000,household reciprocal,3.000,500.000,617.763,Q3,y,1.000,505.520,113.345,0.603,High,3.000,food_secure,NaN,5.000,1.000,NaN,5.230,28.355,"1,010,580.000",0.754,0.246,Diversified rainfed better-off
4,bf_adn_2019_5_1,burkina_faso,bf,2019,NaN,11.200,-1.000,y,1.000,Food Shortage,4.000,8,6.520,f,1.000,primary,0.000,n,4.047,own_land rent_out_land,male_adult,n,0.000,household reciprocal,3.000,"1,700.000",420.079,Q2,n,0.000,34.998,5.368,0.326,Low,7.000,food_secure,NaN,2.000,1.000,NaN,2.120,171.782,"4,206,400.000",0.362,0.638,Low-income low-productivity rainfed


,field,valid_rows,pct_of_dataset
0,food_shortage_flag,47399,86.400
1,nr_months_food_shortage,28537,52.000
2,irrigated_flag,50149,91.400
3,land_productivity_kg_per_ha,47803,87.100
4,income_ppp_per_mae,48787,88.900
5,country_income_rank_pct,50964,92.900
6,fies_yes_count,25072,45.700
7,hfias_frequency_score,6847,12.500
8,vulnerability_profile_k4,41798,76.200


## 6. Export

Write the final Tableau-ready CSV. This is the single file Tableau should load directly.

In [8]:
output_csv = Path("../data/processed/rhomis_tableau_ready.csv")

tableau_df.to_csv(output_csv, index=False)

print("Export complete.")
print(f"CSV path: {output_csv}")
print(f"Final dataset shape: {tableau_df.shape[0]:,} rows x {tableau_df.shape[1]:,} columns")

Export complete.
CSV path: ../data/processed/rhomis_tableau_ready.csv
Final dataset shape: 54,873 rows x 46 columns
